# 08 — Final Summary Dashboard

<div style="padding: 18px 22px; border-radius: 14px; background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 55%, #2563eb 100%); color: white;">
  <h2 style="margin: 0 0 8px 0;">Bayesian Market Signal Analytics</h2>
  <p style="font-size: 16px; margin: 0;">A concise final project dashboard linking the SQL feature pipeline, exploratory findings, Bayesian return/risk models, regime analysis, and portfolio results.</p>
</div>

## 1. Project title and executive summary

This notebook is the final reviewer-facing summary for the project. It is intentionally compact: it rebuilds the SQL layer from the tracked OHLCV data, computes the headline financial diagnostics, and presents posterior-style uncertainty summaries for returns, downside risk, regimes, and portfolio weights.

**Executive summary.** The project studies seven large technology equities (`AAPL`, `AMZN`, `FB`, `GOOG`, `MSFT`, `NFLX`, `NVDA`) using a reproducible pipeline:

1. validate and load historical OHLCV data;
2. create SQL tables/views for clean prices, returns, rolling features, drawdowns, risk summaries, and portfolio inputs;
3. compare cumulative performance, volatility, drawdowns, and correlations;
4. estimate uncertainty around expected returns and downside probabilities;
5. compare equal-weight, traditional optimized, and Bayesian resampled portfolio allocations.

> **Educational disclaimer:** This notebook is for educational and research purposes only. Results are historical, model-dependent, and do not include fundamentals, macro variables, transaction costs, taxes, liquidity constraints, or live market data. Nothing in this notebook should be interpreted as financial advice.


## Setup: imports, styling, paths, and reproducibility

The dashboard is self-contained: it can be run from either the repository root or the `notebooks/` directory, and it falls back to the checked-in `tech_stocks.csv` if the configured raw-data staging path is absent.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, Markdown, display

NOTEBOOK_PATH = Path.cwd().resolve()
ROOT = NOTEBOOK_PATH if (NOTEBOOK_PATH / "src").exists() else NOTEBOOK_PATH.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DUCKDB_PATH, FIGURES_DIR, RAW_CSV_PATH, SQL_DIR, STOCK_SYMBOLS, TRADING_DAYS_PER_YEAR
from src.portfolio import estimate_mean_covariance, optimize_max_sharpe_long_only, portfolio_performance
from src.sql_utils import initialize_database, query_to_df, run_sql_pipeline

RNG = np.random.default_rng(42)
RAW_CSV_FOR_NOTEBOOK = RAW_CSV_PATH if RAW_CSV_PATH.exists() else ROOT / "tech_stocks.csv"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

PALETTE = dict(zip(STOCK_SYMBOLS, sns.color_palette("tab10", n_colors=len(STOCK_SYMBOLS))))

print(f"Repository root: {ROOT}")
print(f"Raw CSV used:     {RAW_CSV_FOR_NOTEBOOK}")
print(f"DuckDB path:      {DUCKDB_PATH}")


## 2. Dataset overview

This section summarizes the raw OHLCV data: symbols, date coverage, row count, and available price/volume fields.


In [ ]:
con = initialize_database(raw_csv_path=RAW_CSV_FOR_NOTEBOOK, db_path=DUCKDB_PATH)
run_sql_pipeline(con)

raw_overview = query_to_df(
    con,
    """
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT symbol) AS symbol_count,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM raw_prices
    """,
)

symbol_coverage = query_to_df(
    con,
    """
    SELECT
        symbol,
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        COUNT(*) AS rows,
        ROUND(MIN(close_adjusted), 2) AS min_adj_close,
        ROUND(MAX(close_adjusted), 2) AS max_adj_close,
        ROUND(AVG(volume), 0) AS avg_volume
    FROM raw_prices
    GROUP BY symbol
    ORDER BY symbol
    """,
)

ohlcv_fields = pd.DataFrame(
    {
        "field": ["open", "high", "low", "close", "close_adjusted", "volume", "split_coefficient"],
        "meaning": [
            "Opening trade price for the day",
            "Highest trade price for the day",
            "Lowest trade price for the day",
            "Raw closing trade price",
            "Split/dividend-adjusted close used for returns",
            "Shares traded during the session",
            "Corporate-action split adjustment coefficient",
        ],
    }
)

display(Markdown("### Dataset snapshot"))
display(raw_overview)
display(Markdown("### Symbol coverage"))
display(symbol_coverage)
display(Markdown("### OHLCV fields"))
display(ohlcv_fields)


In [ ]:
def metric_card(label: str, value: str, note: str = "") -> str:
    return f"""
    <div style='flex:1; min-width:160px; padding:14px 16px; margin:6px; border-radius:14px; background:#f8fafc; border:1px solid #dbeafe;'>
      <div style='font-size:12px; text-transform:uppercase; color:#475569; letter-spacing:.06em;'>{label}</div>
      <div style='font-size:26px; font-weight:750; color:#1e3a8a; margin-top:4px;'>{value}</div>
      <div style='font-size:12px; color:#64748b; margin-top:2px;'>{note}</div>
    </div>
    """

row = raw_overview.iloc[0]
cards = "".join(
    [
        metric_card("Rows", f"{int(row['row_count']):,}", "raw OHLCV records"),
        metric_card("Symbols", f"{int(row['symbol_count'])}", ", ".join(symbol_coverage["symbol"])),
        metric_card("Date coverage", f"{row['first_date']} → {row['last_date']}", "full raw dataset"),
        metric_card("Fields", "7 OHLCV+", "open/high/low/close/adj close/volume/splits"),
    ]
)
display(HTML(f"<div style='display:flex; flex-wrap:wrap;'>{cards}</div>"))


## 3. SQL pipeline summary

The SQL layer turns raw observations into reusable analytical tables/views. The objects below support all downstream notebooks and keep transformations auditable.


In [ ]:
pipeline_summary = pd.DataFrame(
    [
        ("raw_prices", "Source OHLCV table loaded from the validated CSV."),
        ("clean_prices", "Deduplicated, typed, adjusted-close price panel for analysis."),
        ("daily_returns", "Simple and log returns by symbol and date."),
        ("rolling_features", "Rolling averages, volatility, momentum, and drawdown inputs."),
        ("drawdowns", "Peak-to-trough drawdown history by symbol."),
        ("stock_level_risk_summary", "Symbol-level annualized return, volatility, downside, VaR/CVaR, and drawdown metrics."),
        ("portfolio_summary_inputs", "Aligned return summary used for portfolio modeling."),
        ("portfolio_returns_wide", "Wide date-by-symbol return matrix for covariance and optimization."),
        ("return_correlation_matrix_long", "Long-form pairwise return correlations."),
    ],
    columns=["SQL object", "Role in the project"],
)

display(pipeline_summary)

object_counts = []
for object_name in pipeline_summary["SQL object"]:
    count_df = query_to_df(con, f"SELECT COUNT(*) AS rows FROM {object_name}")
    object_counts.append({"SQL object": object_name, "rows": int(count_df.loc[0, "rows"])})
object_counts = pd.DataFrame(object_counts)
display(object_counts)


## 4. Key EDA findings

The EDA compares cumulative returns, annualized volatility, maximum drawdown, and cross-stock correlations. These summaries provide the historical context for the Bayesian and portfolio sections.


## Methodology notes for dashboard metrics

Dashboard figures use the same finance conventions as the research notebooks: log returns for modeling and annualization, simple-return language for intuitive losses, annualized return `= mean daily log return × 252`, annualized volatility `= daily volatility × sqrt(252)`, drawdown `= adj_close / running_peak - 1`, 5% VaR as the historical lower-tail threshold, and 5% CVaR as the average return in the worst 5% of days. Sharpe-like ratios set `risk_free_rate = 0`, so they are descriptive unless a risk-free series is added. Correlation plots should be interpreted cautiously because correlated technology stocks can provide limited diversification during common growth-stock stress. Student-t assumptions in the Bayesian sections address tail risk by allowing more extreme returns than a normal likelihood.



In [ ]:
clean_prices = query_to_df(
    con,
    """
    SELECT symbol, date, adj_close, volume
    FROM clean_prices
    ORDER BY symbol, date
    """,
)
daily_returns = query_to_df(
    con,
    """
    SELECT symbol, date, simple_return, log_return
    FROM daily_returns
    WHERE log_return IS NOT NULL AND simple_return IS NOT NULL
    ORDER BY symbol, date
    """,
)
risk_summary = query_to_df(con, "SELECT * FROM stock_level_risk_summary ORDER BY symbol")
portfolio_returns_wide = query_to_df(con, "SELECT * FROM portfolio_returns_wide ORDER BY date")

clean_prices["date"] = pd.to_datetime(clean_prices["date"])
daily_returns["date"] = pd.to_datetime(daily_returns["date"])
portfolio_returns_wide["date"] = pd.to_datetime(portfolio_returns_wide["date"])

cumulative_returns = (
    daily_returns.sort_values(["symbol", "date"])
    .assign(cumulative_return=lambda df: np.expm1(df.groupby("symbol")["log_return"].cumsum()))
)

eda_summary = (
    daily_returns.groupby("symbol")
    .agg(
        mean_daily_return=("log_return", "mean"),
        annualized_return=("log_return", lambda x: x.mean() * TRADING_DAYS_PER_YEAR),
        annualized_volatility=("log_return", lambda x: x.std(ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)),
        bad_day_rate=("simple_return", lambda x: (x <= -0.02).mean()),
        extreme_loss_rate=("simple_return", lambda x: (x <= -0.05).mean()),
    )
    .reset_index()
)

drawdown_summary = query_to_df(
    con,
    """
    SELECT symbol, MIN(drawdown) AS max_drawdown
    FROM drawdowns
    GROUP BY symbol
    ORDER BY symbol
    """,
)
eda_summary = eda_summary.merge(drawdown_summary, on="symbol", how="left")
eda_summary["return_to_volatility"] = eda_summary["annualized_return"] / eda_summary["annualized_volatility"]

display(eda_summary.sort_values("return_to_volatility", ascending=False))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.lineplot(
    data=cumulative_returns,
    x="date",
    y="cumulative_return",
    hue="symbol",
    palette=PALETTE,
    linewidth=1.7,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Cumulative log-return growth")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Cumulative return")
axes[0, 0].legend(title="", ncol=2, fontsize=8)

sns.barplot(
    data=eda_summary.sort_values("annualized_volatility", ascending=False),
    x="symbol",
    y="annualized_volatility",
    hue="symbol",
    palette=PALETTE,
    legend=False,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Annualized volatility")
axes[0, 1].set_xlabel("Symbol")
axes[0, 1].set_ylabel("Volatility")

sns.barplot(
    data=eda_summary.sort_values("max_drawdown"),
    x="symbol",
    y="max_drawdown",
    hue="symbol",
    palette=PALETTE,
    legend=False,
    ax=axes[1, 0],
)
axes[1, 0].set_title("Maximum drawdown")
axes[1, 0].set_xlabel("Symbol")
axes[1, 0].set_ylabel("Worst peak-to-trough drawdown")

return_matrix = portfolio_returns_wide.set_index("date").loc[:, list(STOCK_SYMBOLS)]
corr = return_matrix.corr()
sns.heatmap(corr, vmin=-1, vmax=1, cmap="vlag", annot=True, fmt=".2f", square=True, ax=axes[1, 1])
axes[1, 1].set_title("Aligned daily-return correlations")
axes[1, 1].set_xlabel("Symbol")
axes[1, 1].set_ylabel("Symbol")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
figure_path = FIGURES_DIR / "08_summary_eda_dashboard.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure: {figure_path.relative_to(ROOT)}")


## 5. Bayesian return findings

The earlier Bayesian return notebook uses a hierarchical Student-t model. For a fast final dashboard refresh, the table below uses posterior draws for each symbol's expected daily log return centered on the observed mean with standard error `s / sqrt(n)`. This preserves the key Bayesian communication pattern: estimates are distributions, not single point values.

Reported metrics:

- **Posterior expected return table:** posterior mean and 90% credible interval for annualized expected return.
- **Probability positive return:** posterior probability that the expected daily return is above zero.
- **Probability best expected return:** share of posterior draws in which a stock has the highest expected return among the seven names.


In [ ]:
def posterior_mean_return_draws(returns_long: pd.DataFrame, draws: int = 20_000) -> pd.DataFrame:
    output = {}
    for symbol, group in returns_long.groupby("symbol"):
        x = group["log_return"].dropna().to_numpy()
        mean = x.mean()
        se = x.std(ddof=1) / np.sqrt(len(x))
        output[symbol] = RNG.normal(mean, se, size=draws)
    return pd.DataFrame(output).loc[:, list(STOCK_SYMBOLS)]

return_draws_daily = posterior_mean_return_draws(daily_returns)
return_draws_annual = return_draws_daily * TRADING_DAYS_PER_YEAR
best_symbol_by_draw = return_draws_daily.idxmax(axis=1)

posterior_return_summary = pd.DataFrame(
    {
        "symbol": return_draws_annual.columns,
        "posterior_expected_return_mean": return_draws_annual.mean().values,
        "credible_interval_5pct": return_draws_annual.quantile(0.05).values,
        "credible_interval_95pct": return_draws_annual.quantile(0.95).values,
        "probability_positive_return": (return_draws_daily > 0).mean().values,
        "probability_best_expected_return": [
            (best_symbol_by_draw == symbol).mean() for symbol in return_draws_daily.columns
        ],
    }
).sort_values("posterior_expected_return_mean", ascending=False)

display(posterior_return_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.8))
plot_df = posterior_return_summary.sort_values("posterior_expected_return_mean")
y = np.arange(len(plot_df))
ax.errorbar(
    plot_df["posterior_expected_return_mean"],
    y,
    xerr=[
        plot_df["posterior_expected_return_mean"] - plot_df["credible_interval_5pct"],
        plot_df["credible_interval_95pct"] - plot_df["posterior_expected_return_mean"],
    ],
    fmt="o",
    color="#1d4ed8",
    ecolor="#93c5fd",
    elinewidth=4,
    capsize=4,
)
ax.axvline(0, color="#334155", linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(plot_df["symbol"])
ax.set_title("Posterior expected annualized return with 90% intervals")
ax.set_xlabel("Annualized expected log return")
ax.set_ylabel("Symbol")
plt.tight_layout()
plt.show()


## 6. Bayesian risk findings

Risk is summarized probabilistically with posterior volatility, bad-day probability, and extreme-loss probability.

- **Posterior volatility:** bootstrap posterior over annualized realized volatility.
- **Bad-day probability:** Beta posterior for `simple_return <= -2%`.
- **Extreme-loss probability:** Beta posterior for `simple_return <= -5%`.


In [ ]:
def bootstrap_volatility_draws(returns_long: pd.DataFrame, draws: int = 5_000) -> pd.DataFrame:
    output = {}
    for symbol, group in returns_long.groupby("symbol"):
        x = group["log_return"].dropna().to_numpy()
        sample_idx = RNG.integers(0, len(x), size=(draws, len(x)))
        sampled = x[sample_idx]
        output[symbol] = sampled.std(axis=1, ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)
    return pd.DataFrame(output).loc[:, list(STOCK_SYMBOLS)]

def beta_event_probability_draws(returns_long: pd.DataFrame, threshold: float, draws: int = 20_000) -> pd.DataFrame:
    output = {}
    for symbol, group in returns_long.groupby("symbol"):
        events = (group["simple_return"] <= threshold).sum()
        non_events = len(group) - events
        output[symbol] = RNG.beta(1 + events, 1 + non_events, size=draws)
    return pd.DataFrame(output).loc[:, list(STOCK_SYMBOLS)]

vol_draws = bootstrap_volatility_draws(daily_returns)
bad_day_draws = beta_event_probability_draws(daily_returns, threshold=-0.02)
extreme_loss_draws = beta_event_probability_draws(daily_returns, threshold=-0.05)

posterior_risk_summary = pd.DataFrame(
    {
        "symbol": list(STOCK_SYMBOLS),
        "posterior_volatility_mean": vol_draws.mean().values,
        "volatility_5pct": vol_draws.quantile(0.05).values,
        "volatility_95pct": vol_draws.quantile(0.95).values,
        "bad_day_probability_mean": bad_day_draws.mean().values,
        "extreme_loss_probability_mean": extreme_loss_draws.mean().values,
    }
).sort_values("posterior_volatility_mean", ascending=False)

display(posterior_risk_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

sns.barplot(
    data=posterior_risk_summary,
    x="symbol",
    y="posterior_volatility_mean",
    hue="symbol",
    palette=PALETTE,
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Posterior annualized volatility")
axes[0].set_xlabel("Symbol")
axes[0].set_ylabel("Volatility")

risk_long = posterior_risk_summary.melt(
    id_vars="symbol",
    value_vars=["bad_day_probability_mean", "extreme_loss_probability_mean"],
    var_name="risk_event",
    value_name="posterior_probability",
)
sns.barplot(data=risk_long, x="symbol", y="posterior_probability", hue="risk_event", ax=axes[1])
axes[1].set_title("Posterior downside event probabilities")
axes[1].set_xlabel("Symbol")
axes[1].set_ylabel("Probability")
axes[1].legend(title="", labels=["Bad day (≤ -2%)", "Extreme loss (≤ -5%)"])
plt.tight_layout()
plt.show()


## 7. Regime findings

The regime notebook models a two-state market process. This dashboard summarizes the same intuition with a transparent realized-volatility split:

- compute an equal-weight market return across all seven stocks;
- estimate a rolling 63-trading-day annualized volatility;
- label observations below the median rolling volatility as **low-volatility regime** and observations above the median as **high-volatility regime**.


In [ ]:
market_returns = return_matrix.dropna().copy()
market_returns["equal_weight_market_return"] = market_returns.mean(axis=1)
market_returns["rolling_63d_volatility"] = (
    market_returns["equal_weight_market_return"].rolling(63).std() * np.sqrt(TRADING_DAYS_PER_YEAR)
)
vol_median = market_returns["rolling_63d_volatility"].median()
market_returns["regime"] = np.where(
    market_returns["rolling_63d_volatility"] <= vol_median,
    "Low-volatility regime",
    "High-volatility regime",
)
regime_ready = market_returns.dropna(subset=["rolling_63d_volatility"])

regime_summary = (
    regime_ready.groupby("regime")
    .agg(
        trading_days=("equal_weight_market_return", "size"),
        annualized_return=("equal_weight_market_return", lambda x: x.mean() * TRADING_DAYS_PER_YEAR),
        regime_volatility=("equal_weight_market_return", lambda x: x.std(ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)),
        worst_daily_return=("equal_weight_market_return", "min"),
    )
    .reset_index()
    .sort_values("regime_volatility")
)

display(regime_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
sns.lineplot(
    data=regime_ready.reset_index(),
    x="date",
    y="rolling_63d_volatility",
    color="#1d4ed8",
    linewidth=1.4,
    ax=ax,
)
ax.axhline(vol_median, color="#dc2626", linestyle="--", linewidth=1.2, label="Median split")
ax.set_title("Equal-weight tech basket: rolling 63-day annualized volatility")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized volatility")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Portfolio findings

This section compares:

1. **Equal weight:** transparent baseline with `1 / N` in each stock.
2. **Traditional optimization:** long-only maximum Sharpe-like allocation using sample means and covariance.
3. **Bayesian optimization:** bootstrap-resampled means/covariances, optimized per posterior draw, then summarized as an allocation distribution.

The Bayesian portfolio output is especially useful because it shows allocation uncertainty rather than presenting a single optimized weight vector as if it were certain.


In [ ]:
asset_returns = return_matrix.dropna().loc[:, list(STOCK_SYMBOLS)]
mean_returns, covariance_matrix = estimate_mean_covariance(asset_returns)

symbols = list(asset_returns.columns)
equal_weights = pd.Series(1 / len(symbols), index=symbols, name="Equal weight")
traditional_weights, traditional_perf = optimize_max_sharpe_long_only(mean_returns, covariance_matrix)
equal_perf = portfolio_performance(equal_weights, mean_returns, covariance_matrix)

bayesian_weight_draws = []
for _ in range(500):
    sample = asset_returns.iloc[RNG.integers(0, len(asset_returns), size=len(asset_returns))]
    sample_mean = sample.mean()
    sample_cov = sample.cov()
    weights, _ = optimize_max_sharpe_long_only(sample_mean, sample_cov)
    bayesian_weight_draws.append(weights.reindex(symbols).to_numpy())

bayesian_weight_draws = pd.DataFrame(bayesian_weight_draws, columns=symbols)
bayesian_mean_weights = bayesian_weight_draws.mean().rename("Bayesian optimization")
bayesian_perf = portfolio_performance(bayesian_mean_weights, mean_returns, covariance_matrix)

portfolio_comparison = pd.DataFrame(
    [
        {"portfolio": "Equal weight", **equal_perf},
        {"portfolio": "Traditional max Sharpe", **traditional_perf},
        {"portfolio": "Bayesian mean optimized weights", **bayesian_perf},
    ]
)

allocation_summary = pd.DataFrame(
    {
        "symbol": symbols,
        "equal_weight": equal_weights.values,
        "traditional_weight": traditional_weights.reindex(symbols).values,
        "bayesian_weight_mean": bayesian_weight_draws.mean().values,
        "bayesian_weight_5pct": bayesian_weight_draws.quantile(0.05).values,
        "bayesian_weight_95pct": bayesian_weight_draws.quantile(0.95).values,
    }
).sort_values("bayesian_weight_mean", ascending=False)

display(portfolio_comparison)
display(allocation_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
weights_long = allocation_summary.melt(
    id_vars="symbol",
    value_vars=["equal_weight", "traditional_weight", "bayesian_weight_mean"],
    var_name="portfolio",
    value_name="weight",
)
sns.barplot(data=weights_long, x="symbol", y="weight", hue="portfolio", ax=axes[0])
axes[0].set_title("Portfolio allocation comparison")
axes[0].set_xlabel("Symbol")
axes[0].set_ylabel("Weight")
axes[0].legend(title="")

ordered_symbols = allocation_summary["symbol"].tolist()
sns.boxplot(data=bayesian_weight_draws.loc[:, ordered_symbols], orient="v", ax=axes[1], color="#93c5fd")
axes[1].set_title("Posterior allocation uncertainty")
axes[1].set_ylabel("Optimized weight across bootstrap posterior draws")
axes[1].set_xlabel("Symbol")
plt.tight_layout()
figure_path = FIGURES_DIR / "08_summary_portfolio_dashboard.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure: {figure_path.relative_to(ROOT)}")


## 9. Final interpretation

The cells below convert the dashboard tables into concise review notes: strongest historical risk-adjusted performers, highest uncertainty, largest downside-risk contributors, and why the Bayesian framing matters.


In [ ]:
best_risk_adjusted = eda_summary.sort_values("return_to_volatility", ascending=False).head(3)["symbol"].tolist()
highest_uncertainty = posterior_risk_summary.sort_values("posterior_volatility_mean", ascending=False).head(3)["symbol"].tolist()
most_downside = posterior_risk_summary.sort_values("bad_day_probability_mean", ascending=False).head(3)["symbol"].tolist()
most_extreme = posterior_risk_summary.sort_values("extreme_loss_probability_mean", ascending=False).head(3)["symbol"].tolist()
most_alloc_uncertainty = (
    bayesian_weight_draws.std()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)

interpretation = f"""
### Interpretation highlights

- **Strongest historical risk-adjusted performance:** {', '.join(best_risk_adjusted)} ranked highest by annualized return divided by annualized volatility in the historical SQL return panel.
- **Highest uncertainty / volatility:** {', '.join(highest_uncertainty)} had the largest posterior annualized volatility estimates.
- **Most downside-risk exposure:** {', '.join(most_downside)} had the highest posterior probability of a daily loss of at least 2%; {', '.join(most_extreme)} ranked highest for daily losses of at least 5%.
- **Most uncertain optimized allocations:** {', '.join(most_alloc_uncertainty)} varied most across Bayesian bootstrap portfolio draws, indicating that their recommended weights are more sensitive to sampling uncertainty.
- **Why Bayesian methods add value here:** daily equity returns are noisy and fat-tailed, so point estimates can overstate certainty. Posterior distributions make uncertainty visible in expected returns, volatility, downside probabilities, regime behavior, and optimized portfolio weights.
"""
display(Markdown(interpretation))


## 10. Limitations

- The analysis uses **historical OHLCV only**.
- It includes **no fundamentals** such as revenue, earnings, margins, valuation ratios, or balance-sheet metrics.
- It includes **no macro variables** such as rates, inflation, unemployment, or liquidity conditions.
- It ignores **transaction costs, taxes, slippage, borrow costs, and market impact**.
- It uses a static historical sample and **no live data**.
- It is an educational, descriptive modeling dashboard and should not be interpreted as guidance to trade or hold any security.


## 11. Future work

High-impact extensions for the next iteration:

1. Add `SPY` and `QQQ` benchmarks for market-relative performance.
2. Add Fama-French factors to separate stock-specific alpha from factor exposure.
3. Add interest rates and inflation to contextualize valuation and discount-rate regimes.
4. Add earnings dates to study event-driven volatility and jump risk.
5. Build a full Bayesian dynamic factor model with time-varying loadings and volatilities.
6. Publish the results as an interactive Streamlit dashboard for non-technical reviewers.


---

### Reviewer checklist

This final notebook should let a reviewer quickly answer:

- What data was used?
- What did the SQL pipeline create?
- Which stocks dominated historical return and risk patterns?
- How uncertain are return, volatility, downside, and allocation estimates?
- How do baseline and optimized portfolios compare?
- What assumptions and limitations matter most?


## Conclusion

The dashboard ties together the validated data layer, DuckDB feature pipeline, descriptive risk metrics, Bayesian uncertainty summaries, and portfolio comparison outputs. Overall, the project is reproducible from `data/raw/tech_stocks.csv` and presents historical technology-stock risk patterns with uncertainty rather than point-estimate certainty.
